In [2]:
%pip install matplotlib seaborn

  Using cached matplotlib-3.11.1-cp314-cp314-win_amd64.whl.metadata (80 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp314-cp314-win_amd64.whl.metadata (121 kB)
  Using cached kiwisolver-1.5.0-cp314-cp314-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.3.0-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached matplotlib-3.11.1-cp314-cp314-win_amd64.whl (9.5 MB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl (232 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.63.0-cp314-cp314-win_amd64.whl (2.3 MB)
Using cached kiwisolver-1.5.0-cp314-cp314-win_amd64.whl (75 kB)
Using cached pillow-12.3.0-cp314-cp314-win_amd64.whl (7.2 MB)
U

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

RAW_PATH = PROJECT_ROOT / "secondary"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
FEATURE_PATH = PROJECT_ROOT / "data" / "processed" / "features"
MODEL_PATH = PROJECT_ROOT / "models"

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
FEATURE_PATH.mkdir(parents=True, exist_ok=True)
MODEL_PATH.mkdir(parents=True, exist_ok=True)

print("Raw:", RAW_PATH)
print("Features:", FEATURE_PATH)
print("Models:", MODEL_PATH)

Raw: c:\Users\DELL\Desktop\CTS_hackathon_demo\secondary
Features: c:\Users\DELL\Desktop\CTS_hackathon_demo\data\processed\features
Models: c:\Users\DELL\Desktop\CTS_hackathon_demo\models


In [5]:
files = list(RAW_PATH.glob("*.csv"))

print("Files found:", len(files))

for f in files:
    print(f.name)

Files found: 4
Train-1542865627584.csv
Train_Beneficiarydata-1542865627584.csv
Train_Inpatientdata-1542865627584.csv
Train_Outpatientdata-1542865627584.csv


In [6]:
beneficiary_file = next(
    f for f in files
    if "Beneficiary" in f.name
)

inpatient_file = next(
    f for f in files
    if "Inpatient" in f.name
)

outpatient_file = next(
    f for f in files
    if "Outpatient" in f.name
)

provider_file = next(
    f for f in files
    if (
        "Beneficiary" not in f.name
        and "Inpatient" not in f.name
        and "Outpatient" not in f.name
    )
)

print("Beneficiary :", beneficiary_file.name)
print("Inpatient   :", inpatient_file.name)
print("Outpatient  :", outpatient_file.name)
print("Provider    :", provider_file.name)

Beneficiary : Train_Beneficiarydata-1542865627584.csv
Inpatient   : Train_Inpatientdata-1542865627584.csv
Outpatient  : Train_Outpatientdata-1542865627584.csv
Provider    : Train-1542865627584.csv


In [7]:
beneficiary = pd.read_csv(
    beneficiary_file,
    low_memory=False
)

inpatient = pd.read_csv(
    inpatient_file,
    low_memory=False
)

outpatient = pd.read_csv(
    outpatient_file,
    low_memory=False
)

provider_labels = pd.read_csv(
    provider_file,
    low_memory=False
)

print("Beneficiary :", beneficiary.shape)
print("Inpatient   :", inpatient.shape)
print("Outpatient  :", outpatient.shape)
print("Labels      :", provider_labels.shape)

Beneficiary : (138556, 25)
Inpatient   : (40474, 30)
Outpatient  : (517737, 27)
Labels      : (5410, 2)


In [8]:
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.replace(" ", "_")
    )
    return df

beneficiary = clean_columns(beneficiary)
inpatient = clean_columns(inpatient)
outpatient = clean_columns(outpatient)
provider_labels = clean_columns(provider_labels)

print("Column names cleaned.")

Column names cleaned.


In [9]:
datasets = {
    "Beneficiary": beneficiary,
    "Inpatient": inpatient,
    "Outpatient": outpatient,
    "Provider Labels": provider_labels
}

for name, df in datasets.items():
    before = len(df)
    df.drop_duplicates(inplace=True)
    after = len(df)

    print(
        f"{name}: removed {before - after} duplicates"
    )

Beneficiary: removed 0 duplicates
Inpatient: removed 0 duplicates
Outpatient: removed 0 duplicates
Provider Labels: removed 0 duplicates


In [10]:
for df in [
    beneficiary,
    inpatient,
    outpatient
]:
    
    for col in ["BeneID", "Provider", "ClaimID"]:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype("string")
                .str.strip()
            )

provider_labels["Provider"] = (
    provider_labels["Provider"]
    .astype("string")
    .str.strip()
)

print("IDs cleaned.")

IDs cleaned.


In [11]:
beneficiary_date_cols = ["DOB", "DOD"]

for col in beneficiary_date_cols:
    if col in beneficiary.columns:
        beneficiary[col] = pd.to_datetime(
            beneficiary[col],
            errors="coerce"
        )

inpatient_date_cols = [
    "ClaimStartDt",
    "ClaimEndDt",
    "AdmissionDt",
    "DischargeDt"
]

for col in inpatient_date_cols:
    if col in inpatient.columns:
        inpatient[col] = pd.to_datetime(
            inpatient[col],
            errors="coerce"
        )

outpatient_date_cols = [
    "ClaimStartDt",
    "ClaimEndDt"
]

for col in outpatient_date_cols:
    if col in outpatient.columns:
        outpatient[col] = pd.to_datetime(
            outpatient[col],
            errors="coerce"
        )

print("Dates converted.")

Dates converted.


In [12]:
REFERENCE_DATE = pd.Timestamp("2010-12-31")

beneficiary["Age"] = (
    (REFERENCE_DATE - beneficiary["DOB"])
    .dt.days / 365.25
)

beneficiary.loc[
    (beneficiary["Age"] < 0) |
    (beneficiary["Age"] > 120),
    "Age"
] = np.nan

print(beneficiary["Age"].describe())

count    138556.000000
mean         74.667492
std          12.727620
min          27.082820
25%          69.081451
50%          75.331964
75%          82.997947
max         101.995893
Name: Age, dtype: float64


In [13]:
chronic_cols = [
    c for c in beneficiary.columns
    if c.lower().startswith("chroniccond")
]

print(chronic_cols)

['ChronicCond_Alzheimer', 'ChronicCond_Heartfailure', 'ChronicCond_KidneyDisease', 'ChronicCond_Cancer', 'ChronicCond_ObstrPulmonary', 'ChronicCond_Depression', 'ChronicCond_Diabetes', 'ChronicCond_IschemicHeart', 'ChronicCond_Osteoporasis', 'ChronicCond_rheumatoidarthritis', 'ChronicCond_stroke']


In [14]:
for col in chronic_cols:
    beneficiary[col] = pd.to_numeric(
        beneficiary[col],
        errors="coerce"
    )

    beneficiary[col] = beneficiary[col].map({
        1: 1,
        2: 0
    })

beneficiary["ChronicConditionCount"] = (
    beneficiary[chronic_cols]
    .sum(axis=1)
)

print(
    beneficiary[
        ["BeneID", "Age", "ChronicConditionCount"]
    ].head()
)

      BeneID        Age  ChronicConditionCount
0  BENE11001  67.997262                      7
1  BENE11002  74.329911                      0
2  BENE11003  74.414784                      2
3  BENE11004  88.501027                      6
4  BENE11005  75.331964                      2


In [15]:
ip_bene_provider = inpatient[
    ["Provider", "BeneID"]
].drop_duplicates()

op_bene_provider = outpatient[
    ["Provider", "BeneID"]
].drop_duplicates()

provider_bene = pd.concat(
    [
        ip_bene_provider,
        op_bene_provider
    ],
    ignore_index=True
).drop_duplicates()

print(provider_bene.shape)

(363300, 2)


(363300, 2)


In [17]:
provider_bene = provider_bene.merge(
    beneficiary,
    on="BeneID",
    how="left"
)

print(provider_bene.shape)

(363300, 28)


In [18]:
beneficiary_features = (
    provider_bene
    .groupby("Provider")
    .agg(
        Unique_Beneficiaries=("BeneID", "nunique"),
        Avg_Patient_Age=("Age", "mean"),
        Avg_Chronic_Conditions=(
            "ChronicConditionCount",
            "mean"
        )
    )
    .reset_index()
)

beneficiary_features.head()

,Provider,Unique_Beneficiaries,Avg_Patient_Age,Avg_Chronic_Conditions
0,PRV51001,24,79.669062,5.541667
1,PRV51003,117,70.472981,4.367521
2,PRV51004,138,73.945402,4.318841
3,PRV51005,495,71.516855,3.884848
4,PRV51007,58,69.485898,3.879310


In [19]:
ip_features = (
    inpatient
    .groupby("Provider")
    .agg(
        IP_Claim_Count=("ClaimID", "nunique"),
        IP_Unique_Beneficiaries=(
            "BeneID",
            "nunique"
        ),
        IP_Total_Reimbursement=(
            "InscClaimAmtReimbursed",
            "sum"
        ),
        IP_Avg_Reimbursement=(
            "InscClaimAmtReimbursed",
            "mean"
        ),
        IP_Max_Reimbursement=(
            "InscClaimAmtReimbursed",
            "max"
        ),
        IP_Total_Deductible=(
            "DeductibleAmtPaid",
            "sum"
        ),
        IP_Avg_Deductible=(
            "DeductibleAmtPaid",
            "mean"
        )
    )
    .reset_index()
)

ip_features.head()

,Provider,IP_Claim_Count,IP_Unique_Beneficiaries,IP_Total_Reimbursement,IP_Avg_Reimbursement,IP_Max_Reimbursement,IP_Total_Deductible,IP_Avg_Deductible
0,PRV51001,5,5,97000,19400.000000,42000,5340.0,1068.0
1,PRV51003,62,53,573000,9241.935484,57000,66216.0,1068.0
2,PRV51007,3,3,19000,6333.333333,10000,3204.0,1068.0
3,PRV51008,2,2,25000,12500.000000,21000,2136.0,1068.0
4,PRV51011,1,1,5000,5000.000000,5000,1068.0,1068.0


In [20]:
if (
    "ClaimStartDt" in inpatient.columns
    and "ClaimEndDt" in inpatient.columns
):

    inpatient["ClaimDurationDays"] = (
        inpatient["ClaimEndDt"]
        - inpatient["ClaimStartDt"]
    ).dt.days

    inpatient.loc[
        inpatient["ClaimDurationDays"] < 0,
        "ClaimDurationDays"
    ] = np.nan

    duration_features = (
        inpatient
        .groupby("Provider")
        .agg(
            IP_Avg_Claim_Duration=(
                "ClaimDurationDays",
                "mean"
            ),
            IP_Max_Claim_Duration=(
                "ClaimDurationDays",
                "max"
            )
        )
        .reset_index()
    )

    ip_features = ip_features.merge(
        duration_features,
        on="Provider",
        how="left"
    )

In [21]:
op_features = (
    outpatient
    .groupby("Provider")
    .agg(
        OP_Claim_Count=("ClaimID", "nunique"),
        OP_Unique_Beneficiaries=(
            "BeneID",
            "nunique"
        ),
        OP_Total_Reimbursement=(
            "InscClaimAmtReimbursed",
            "sum"
        ),
        OP_Avg_Reimbursement=(
            "InscClaimAmtReimbursed",
            "mean"
        ),
        OP_Max_Reimbursement=(
            "InscClaimAmtReimbursed",
            "max"
        ),
        OP_Total_Deductible=(
            "DeductibleAmtPaid",
            "sum"
        ),
        OP_Avg_Deductible=(
            "DeductibleAmtPaid",
            "mean"
        )
    )
    .reset_index()
)

op_features.head()

,Provider,OP_Claim_Count,OP_Unique_Beneficiaries,OP_Total_Reimbursement,OP_Avg_Reimbursement,OP_Max_Reimbursement,OP_Total_Deductible,OP_Avg_Deductible
0,PRV51001,20,19,7640,382.000000,1500,0,0.000000
1,PRV51003,70,66,32670,466.714286,3300,70,1.000000
2,PRV51004,149,138,52170,350.134228,3300,310,2.080537
3,PRV51005,1165,495,280910,241.124464,4080,3700,3.175966
4,PRV51007,69,56,14710,213.188406,3300,60,0.869565


In [22]:
diagnosis_cols = [
    c for c in inpatient.columns
    if "DiagnosisCode" in c
]

procedure_cols = [
    c for c in inpatient.columns
    if "ProcedureCode" in c
]

print("Diagnosis columns:", len(diagnosis_cols))
print("Procedure columns:", len(procedure_cols))

Diagnosis columns: 11
Procedure columns: 6


In [23]:
op_diagnosis_cols = [
    c for c in outpatient.columns
    if "DiagnosisCode" in c
]

op_procedure_cols = [
    c for c in outpatient.columns
    if "ProcedureCode" in c
]

print("OP Diagnosis:", len(op_diagnosis_cols))
print("OP Procedure:", len(op_procedure_cols))

OP Diagnosis: 11
OP Procedure: 6


In [24]:
ip_diag_long = inpatient[
    ["Provider"] + diagnosis_cols
].melt(
    id_vars="Provider",
    value_name="DiagnosisCode"
).dropna(subset=["DiagnosisCode"])

ip_diag_features = (
    ip_diag_long
    .groupby("Provider")["DiagnosisCode"]
    .nunique()
    .reset_index(
        name="IP_Unique_Diagnosis_Codes"
    )
)

In [25]:
ip_proc_long = inpatient[
    ["Provider"] + procedure_cols
].melt(
    id_vars="Provider",
    value_name="ProcedureCode"
).dropna(subset=["ProcedureCode"])

ip_proc_features = (
    ip_proc_long
    .groupby("Provider")["ProcedureCode"]
    .nunique()
    .reset_index(
        name="IP_Unique_Procedure_Codes"
    )
)

In [26]:
ip_features = ip_features.merge(
    ip_diag_features,
    on="Provider",
    how="left"
)

ip_features = ip_features.merge(
    ip_proc_features,
    on="Provider",
    how="left"
)

In [27]:
provider_features = beneficiary_features.merge(
    ip_features,
    on="Provider",
    how="outer"
)

provider_features = provider_features.merge(
    op_features,
    on="Provider",
    how="outer"
)

print(
    "Provider feature shape:",
    provider_features.shape
)

Provider feature shape: (5410, 22)


In [28]:
provider_features["Total_Claims"] = (
    provider_features["IP_Claim_Count"].fillna(0)
    +
    provider_features["OP_Claim_Count"].fillna(0)
)

provider_features["Total_Reimbursement"] = (
    provider_features["IP_Total_Reimbursement"].fillna(0)
    +
    provider_features["OP_Total_Reimbursement"].fillna(0)
)

provider_features["Total_Deductible"] = (
    provider_features["IP_Total_Deductible"].fillna(0)
    +
    provider_features["OP_Total_Deductible"].fillna(0)
)

In [29]:
provider_features["Claims_Per_Beneficiary"] = (
    provider_features["Total_Claims"]
    /
    provider_features["Unique_Beneficiaries"]
    .replace(0, np.nan)
)

provider_features["Reimbursement_Per_Beneficiary"] = (
    provider_features["Total_Reimbursement"]
    /
    provider_features["Unique_Beneficiaries"]
    .replace(0, np.nan)
)

In [30]:
provider_features["IP_Claim_Share"] = (
    provider_features["IP_Claim_Count"].fillna(0)
    /
    provider_features["Total_Claims"]
    .replace(0, np.nan)
)

provider_features["OP_Claim_Share"] = (
    provider_features["OP_Claim_Count"].fillna(0)
    /
    provider_features["Total_Claims"]
    .replace(0, np.nan)
)

In [31]:
provider_features["IP_Reimbursement_Range"] = (
    provider_features["IP_Max_Reimbursement"].fillna(0)
    -
    provider_features["IP_Avg_Reimbursement"].fillna(0)
)

provider_features["OP_Reimbursement_Range"] = (
    provider_features["OP_Max_Reimbursement"].fillna(0)
    -
    provider_features["OP_Avg_Reimbursement"].fillna(0)
)

In [32]:
provider_labels["PotentialFraud"] = (
    provider_labels["PotentialFraud"]
    .astype("string")
    .str.strip()
)

provider_features = provider_features.merge(
    provider_labels[
        ["Provider", "PotentialFraud"]
    ],
    on="Provider",
    how="inner"
)

print(provider_features.shape)
print(
    provider_features["PotentialFraud"]
    .value_counts()
)

(5410, 32)
PotentialFraud
No     4904
Yes     506
Name: count, dtype: Int64


In [33]:
print(
    "Unique providers:",
    provider_features["Provider"].nunique()
)

print(
    "Rows:",
    len(provider_features)
)

Unique providers: 5410
Rows: 5410


In [34]:
provider_features.to_csv(
    FEATURE_PATH / "provider_features_v2.csv",
    index=False
)

print(
    "Saved:",
    FEATURE_PATH / "provider_features_v2.csv"
)

Saved: c:\Users\DELL\Desktop\CTS_hackathon_demo\data\processed\features\provider_features_v2.csv
